# Large Language Models

Die meisten modernen Sprachmodelle basieren auf der Transformer-Architektur. Während die Architektur sich stark von einfacheren "klassischen" Modellen, wie N-Grams unterscheidet, so ist die Aufgabe jedoch im Kern dieselbe: gegeben eines Kontexts, was ist die wahrscheinlichste Fortsetzung? Transformer-basierte Sprachmodelle lösen einige der Probleme früherer Methoden, wie beispielsweise das Problem langer Kontexte. Da die Architektur ohne *Rekurrenz* auskommt und rein auf den **Attention**-Mechanismus setzt, können moderne LLMs mit unglaublich langen Kontexten klarkommen.

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    if not os.path.exists("/content/workshop-bundeskonferenz"):
        !git clone -q --depth 1 https://github.com/VoErik/workshop-bundeskonferenz.git /content/workshop-bundeskonferenz
    %cd /content/workshop-bundeskonferenz
    %pip install -q .

%env HF_HUB_DISABLE_XET=1
from workshop.llm import (Sprachmodell, MODELL_1, MODELL_2)
from workshop.viz import visualisiere_sampling

## Nicht alle Modelle sind gleich...

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Aufgabe:** Wir laden zwei Sprachmodelle und stellen beiden dieselben Prompts. Führen Sie die folgenden Zellen aus und beobachten Sie genau, wie die beiden Modelle reagieren.

</div>

### Unser erstes Sprachmodell

In [ ]:
modell_1 = Sprachmodell(MODELL_1)

In [ ]:
modell_1.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_1.prompt(prompt="What is the Pythagorean theorem?")

### Ein anderes Modell

In [ ]:
modell_2 = Sprachmodell(MODELL_2)

In [ ]:
modell_2.prompt(prompt="Isaac Newton discovered that gravity")

In [ ]:
modell_2.prompt(prompt="What is the Pythagoeran theorem?")

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Frage:** Welche Unterschiede fallen Ihnen zwischen `modell_1` und `modell_2` auf? Wie reagieren die beiden auf den Satzanfang, wie auf die Frage? Und stimmen die Inhalte? Haben Sie eine Vermutung, woran die Unterschiede liegen könnten?

</div>

<div style="background-color:#fff3e0; color:#663c00; border:1px solid #ffb74d; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Antwort</summary>

`modell_1` ist ein **Basismodell**. Es wurde ausschließlich darauf trainiert, Texte fortzusetzen. Einen Prompt versteht es nicht als Auftrag, sondern als Anfang eines Dokuments. Den Satzanfang setzt es deshalb meist flüssig fort. Die Frage beantwortet es dagegen oft nicht, sondern schreibt z.B. weitere Fragen – im Internet folgt auf eine Frage eben häufig die nächste, etwa auf Aufgabenblättern oder FAQ-Seiten.

`modell_2` ist ein **Chat-Modell**. Es ist aus einem Basismodell entstanden, das in einem zweiten Schritt mit vielen Beispieldialogen weitertrainiert wurde. Es behandelt jeden Prompt als Nachricht, auf die es antwortet – auch den unvollständigen Satz über Newton.

Beide Modelle funktionieren nach demselben Prinzip: Sie sagen das jeweils nächste Token vorher. Der entscheidende Unterschied liegt im Training. Was dieses zusätzliche Training genau verändert, schauen wir uns im Kapitel *Die Chat-Illusion* an.

Zum Inhalt: Kleine Modelle formulieren sehr überzeugend, liegen fachlich aber nicht immer richtig.

</details>
</div>

#### Und auf Deutsch?

In [ ]:
modell_2.prompt(prompt="Was ist der Satz des Pythagoras?")

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Frage:** Wie gut gelingt die Antwort auf Deutsch? Woran könnte das liegen?

</div>

<div style="background-color:#fff3e0; color:#663c00; border:1px solid #ffb74d; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Antwort</summary>

Ein Sprachmodell kann nur, was in seinen Trainingsdaten vorkommt. Unsere Modelle wurden fast ausschließlich mit englischen Texten trainiert, deshalb sind Grammatik, Wortschatz und Fachwissen auf Deutsch deutlich schwächer. Große kommerzielle Modelle wurden mit Texten in sehr vielen Sprachen trainiert – aber auch dort ist Englisch meist die stärkste Sprache.

Dazu kommt der **Tokenizer**, der ebenfalls an englischen Texten optimiert wurde. Häufige englische Wörter sind oft ein einziges Token, deutsche Wörter werden dagegen häufig in mehrere Stücke zerlegt. Probieren Sie es aus:

```python
print(modell_2.tokenizer.tokenize("The acceleration of the car"))
print(modell_2.tokenizer.tokenize("Die Beschleunigung des Autos"))
```

</details>
</div>

## In-Kontext Lernen | Few Shot Prompting

Im ersten Kapitel hat das Basismodell Fragen nicht beantwortet, sondern einfach Text fortgesetzt. Lässt es sich trotzdem dazu bringen, eine bestimmte Aufgabe zu erledigen?

In [ ]:
basis_modell = modell_1
chat_modell = modell_2

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Aufgabe:** Schreiben Sie einen Prompt, der das **Basismodell** dazu bringt, Fachbegriffe aus der Physik vom Englischen ins Deutsche zu übersetzen.

Mit `max_tokens=3` darf das Modell nur wenige Tokens erzeugen. So sehen Sie nur die Übersetzung und nicht, was das Modell danach noch alles schreibt.

</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Tipp</summary>

Probieren Sie zunächst eine direkte Anweisung wie „Translate into German: voltage“. Klappt das nicht, erinnern Sie sich daran, wie das Basismodell im ersten Kapitel reagiert hat: In welcher Art von Dokument stehen englische und deutsche Fachbegriffe nebeneinander? Ihr Prompt sollte aussehen wie der Anfang eines solchen Dokuments – und genau dort enden, wo die gesuchte Übersetzung beginnt.

</details>
</div>

In [ ]:
prompt = "Translate into German: voltage"

basis_modell.prompt(prompt=sol, max_tokens=120)

In [ ]:
basis_modell.zeige_prompt(sol)

<div style="background-color:#fff3e0; color:#663c00; border:1px solid #ffb74d; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Lösungsvorschlag</summary>

```
"""Task: Translate physics terms from English to German.
English: velocity -> German: Geschwindigkeit
English: acceleration -> German: Beschleunigung
English: force -> German: Kraft
English: voltage -> German:"""
```

Der Prompt endet direkt nach „German:“, das wahrscheinlichste nächste Token ist also der Anfang der Übersetzung. Achten Sie darauf, dass am Ende kein Leerzeichen steht: Bei den meisten Tokenizern gehört das Leerzeichen zum nächsten Wort-Token (z.B. „ Spannung). Ein Leerzeichen am Ende des Prompts bringt das Modell deshalb leicht aus dem Tritt.

</details>
</div>

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Zum Ausprobieren:**

- Was passiert mit nur einem Beispiel? Und ganz ohne?
- Was passiert, wenn die Beispiele uneinheitlich formatiert sind (mal mit Doppelpunkt, mal ohne)?
- Funktioniert es auch für Begriffe aus Chemie oder Biologie? Oder in die andere Richtung?
- Was passiert mit `max_tokens=30`?

</div>

<div style="background-color:#e3f2fd; color:#0d3c61; border:1px solid #90caf9; border-radius:4px; padding:10px 15px; margin:10px 0;">

**Frage:** Das Basismodell kann keine Anweisungen befolgen. Warum funktioniert der Prompt mit Beispielen trotzdem? Was haben Ihre Experimente gezeigt?

</div>

<div style="background-color:#fff3e0; color:#663c00; border:1px solid #ffb74d; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Antwort</summary>

Basismodelle befolgen keine Anweisungen, aber sie sind sehr gut darin, **Muster fortzusetzen**. Im Internet gibt es unzählige Listen, Tabellen, Glossare und Vokabelverzeichnisse. Das Modell hat dadurch gelernt: Hat ein Text eine klare Struktur, geht er meist in derselben Struktur weiter.

Statt dem Modell zu sagen, *was* es tun soll, zeigt man ihm also einige Beispiele und lässt es das Muster fortsetzen. Das nennt man **Few-Shot Prompting** (engl. *shot*, hier: Beispiel):

- **Zero-Shot:** nur die Aufgabe, kein Beispiel
- **One-Shot:** ein Beispiel
- **Few-Shot:** mehrere Beispiele, typischerweise 2 bis 10

Das Bemerkenswerte daran: Das Modell „lernt“ die Aufgabe allein aus dem Prompt, ohne dass sich an seinen Parametern etwas ändert. Man spricht von **In-Kontext Lernen**. Sobald der Prompt weg ist, ist auch das Gelernte wieder verschwunden.

Bekannt wurde das Prinzip 2020 mit dem Artikel *„Language Models are Few-Shot Learners“*, der GPT-3 vorstellte. Er zeigte, dass ein ausreichend großes Basismodell viele Aufgaben allein durch Beispiele im Prompt lösen kann, ganz ohne zusätzliches Training. Einen Teil des Mechanismus dahinter machen wir im Kapitel *Attention* sichtbar: die **Induction Heads**, die Muster im Prompt erkennen und fortsetzen.

</details>
</div>

## Die Chat-Illusion & Instruktions-Tuning

Ein Chat-Modell fühlt sich an wie ein Gesprächspartner. Aber wie viel davon ist echt? Wir stellen dem Modell eine Frage und danach eine Folgefrage, die sich auf die erste bezieht.

In [ ]:
chat_modell.prompt(prompt="Explain the Pythagorean theorem in one sentence.")

In [ ]:
chat_modell.prompt(prompt="Now give me only the formula.")

**Was ist hier passiert?** Das Modell weiß nicht, welche Formel gemeint ist. Der Grund: Jeder Aufruf von `prompt` ist vollkommen unabhängig vom vorherigen. Das Modell selbst hat **kein Gedächtnis**. Es sieht immer nur genau den Text, den wir ihm in diesem Moment übergeben – und dort steht nur „Now give me only the formula.“

Die Methode `prompt_auto` macht es anders: Sie merkt sich den bisherigen Gesprächsverlauf und schickt ihn bei jedem Aufruf **vollständig** mit.

In [ ]:
chat_modell.prompt_auto(prompt="Explain the Pythagorean theorem in one sentence.")

In [ ]:
chat_modell.prompt_auto(prompt="Now give me only the formula.")

### Wie „erinnern“ sich Sprachmodelle?

Das „Gedächtnis“ ist nichts weiter als eine Liste aller bisherigen Nachrichten, die bei jeder neuen Anfrage komplett an das Modell geschickt wird. Genauso funktionieren auch ChatGPT, Claude & Co.: Mit jeder neuen Nachricht wird das gesamte bisherige Gespräch erneut verarbeitet. So sieht unser Gedächtnis gerade aus:

In [ ]:
chat_modell.gedaechtnis.zeige()

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Das Kontextfenster</summary>

Weil das ganze Gespräch jedes Mal neu verarbeitet wird, gibt es eine Obergrenze: das **Kontextfenster**, also die maximale Anzahl an Tokens, die ein Modell auf einmal verarbeiten kann. Bei kleinen Modellen wie SmolLM2 sind das einige Tausend Tokens, große Modelle schaffen Hunderttausende bis über eine Million. Wird ein Gespräch zu lang, muss die Anwendung ältere Teile abschneiden oder zusammenfassen – das Modell „vergisst“ dann den Anfang.

Daraus folgt auch: Jede Antwort wird aufwendiger, je länger das Gespräch ist. Wer für ein neues Thema einen neuen Chat beginnt, spart Rechenzeit und bekommt oft auch bessere Antworten, weil kein unpassender Kontext ablenkt.

</details>
</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Und das „Langzeitgedächtnis“ mancher Chat-Apps?</summary>

Einige Anwendungen werben damit, sich über Gespräche hinweg an Sie zu erinnern. Technisch funktioniert das nach demselben Prinzip: Die Anwendung speichert Notizen über frühere Gespräche und fügt sie bei neuen Anfragen in den Prompt ein. Das Modell selbst verändert sich dabei nicht.

</details>
</div>

### Was sieht das Modell tatsächlich? 

Die Liste der Nachrichten wird vor der Verarbeitung mit einem **Chat-Template** in einen einzigen langen Text umgewandelt. Spezielle Tokens markieren dabei, wer gerade spricht: `<|im_start|>` leitet eine Nachricht ein, gefolgt von der Rolle (`system`, `user` oder `assistant`), und `<|im_end|>` beendet sie.

In [ ]:
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

Achten Sie auf zwei Dinge:

1. **Es gibt einen System-Prompt**, obwohl wir keinen geschrieben haben. Das Template fügt automatisch einen Standardtext ein.
2. **Der Text endet mit `<|im_start|>assistant`.** Das Modell tut nun, was es immer tut: Es setzt den Text fort, Token für Token. Es schreibt also die nächste Nachricht des Assistenten und hört auf, sobald es selbst das Token `<|im_end|>` erzeugt.

Das ist die **Chat-Illusion**: Ein Chat ist für das Modell nur ein Dokument mit einer bestimmten Struktur, das es fortsetzt – genau wie das Basismodell.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Was ist am Chat-Modell also anders?</summary>

Technisch nichts Grundsätzliches: Auch das Chat-Modell sagt nur das nächste Token vorher. Der Unterschied liegt im Training. Beim Instruktions-Tuning hat es sehr viele Dokumente in genau diesem Format gesehen und dabei gelernt: Nach `<|im_start|>assistant` folgt eine hilfreiche Antwort auf die vorherige `user`-Nachricht, und danach kommt `<|im_end|>`.

Dieses Format heißt **ChatML**. Andere Modellfamilien verwenden andere Templates mit anderen Sondertokens, das Prinzip ist aber überall dasselbe.

</details>
</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Was passiert, wenn das Modell nicht aufhört?</summary>

Würde die Anwendung das `<|im_end|>` ignorieren, schriebe das Modell einfach weiter: oft ein neues `<|im_start|>user`, gefolgt von einer erfundenen Frage des Nutzers – und dann der passenden Antwort. Das Modell setzt eben das ganze Dokument fort. Dass es nur „seine“ Rolle spielt, liegt allein an der Anwendung, die an der richtigen Stelle abbricht.

</details>
</div>

## Decoding Parameter

Generieren Sprachmodelle immer dasselbe? Das hängt davon ab, wie wir aus den Wahrscheinlichkeiten, die das Modell berechnet, das nächste Token **auswählen**. Diesen Schritt nennt man **Decoding**. Zwei Stellschrauben schauen wir uns an:

- Die **Temperatur** bestimmt, wie stark der Zufall mitspielt.
- Das **Nukleus-Sampling** (*top-p*) legt fest, welche Tokens überhaupt infrage kommen.

Dafür lassen wir das Modell dreimal hintereinander eine kreative Metapher für eine quadratische Gleichung schreiben, jeweils mit unterschiedlicher Temperatur.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Temperatur – der Name kommt aus der Physik</summary>

Das Modell berechnet für jedes Token $i$ zunächst eine Zahl $z_i$ (den *Logit*). Daraus werden mit der Softmax-Funktion Wahrscheinlichkeiten:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

Kommt Ihnen das bekannt vor? Es ist die **Boltzmann-Verteilung** $p_i \propto e^{-E_i / k_B T}$ aus der statistischen Physik, wobei die Logits die Rolle negativer Energien spielen. Und wie dort gilt:

- **Niedrige Temperatur:** Fast die gesamte Wahrscheinlichkeit liegt beim „Grundzustand“, dem wahrscheinlichsten Token. Für $T \to 0$ wird immer dieses Token gewählt.
- **Hohe Temperatur:** Die Verteilung wird flacher, auch unwahrscheinliche Tokens („angeregte Zustände“) kommen zum Zug.
- **$T = 1$** entspricht den unveränderten Wahrscheinlichkeiten des Modells.

</details>
</div>

In [ ]:
prompt = "Write a creative metaphor for a quadratic equation."
max_tokens = 64

### Temperatur = 0.0

Bei Temperatur 0 wählt das Modell **immer das wahrscheinlichste Token** (*greedy decoding*). Dreimal derselbe Prompt sollte also dreimal exakt dieselbe Antwort ergeben. Die „beste“ Antwort ist das übrigens nicht automatisch: Immer nur das wahrscheinlichste Token zu nehmen, führt oft zu farblosen Texten oder Wiederholungen.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Ist Temperatur 0 wirklich deterministisch?</summary>

In unserem Notebook ja. Bei großen Anbietern erhält man bei Temperatur 0 trotzdem manchmal unterschiedliche Antworten. Der Grund liegt in der Hardware: Grafikkarten rechnen mit Gleitkommazahlen, bei denen die Reihenfolge von Additionen das Ergebnis minimal verändert – und diese Reihenfolge hängt z.B. davon ab, wie viele Anfragen gerade gleichzeitig bearbeitet werden. Liegen zwei Tokens fast gleichauf, kann ein solcher winziger Unterschied den Ausschlag geben. Ab da verläuft der restliche Text anders.

</details>
</div>

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=0.0)

### Temperatur = 0.7

Jetzt wird **gewürfelt**: Das nächste Token wird zufällig gezogen, gewichtet nach seiner Wahrscheinlichkeit. Die drei Antworten unterscheiden sich, bleiben aber sinnvoll. Werte in diesem Bereich sind in vielen Chat-Anwendungen üblich.

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=0.7)

### Temperatur = 1.8

Bei hoher Temperatur bekommen auch sehr unwahrscheinliche Tokens eine echte Chance. Beobachten Sie, wie die Texte zunehmend zerfallen. Das Problem verstärkt sich selbst: Hat das Modell einmal ein unpassendes Token gewählt, muss es den Text darauf aufbauend fortsetzen – Fehler häufen sich an.

In [ ]:
for i in range(3):
    chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=1.8)

### Temperatur & Nukleus-Sampling Visualisiert

Von den rund 49.000 Tokens im Vokabular passen in jedem Moment fast alle überhaupt nicht. Einzeln haben sie winzige Wahrscheinlichkeiten, zusammen ergeben sie aber einen spürbaren Anteil – und genau aus diesem „langen Schwanz“ zieht das Modell bei hoher Temperatur seinen Unsinn.

**Nukleus-Sampling** (*top-p*) schneidet diesen Schwanz ab: Die Tokens werden nach Wahrscheinlichkeit sortiert, und es kommen nur so viele infrage, bis ihre Wahrscheinlichkeiten zusammen den Wert $p$ erreichen (z.B. $p = 0{,}9$, also 90 %). Nur aus diesem „Kern“ (*nucleus*) wird gezogen.

In der folgenden Visualisierung können Sie beide Parameter verändern und beobachten, wie sich die Wahrscheinlichkeitsverteilung verändert.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Temperatur und top-p kombinieren</summary>

In der Praxis werden beide meist zusammen eingesetzt: Die Temperatur macht die Verteilung flacher oder spitzer, top-p sorgt anschließend dafür, dass trotzdem nur plausible Tokens gezogen werden. So lässt sich eine etwas höhere Temperatur für mehr Abwechslung verwenden, ohne dass der Text zerfällt. Bei $p = 1$ ist Nukleus-Sampling ausgeschaltet. Probieren Sie es auch direkt aus:

```python
chat_modell.prompt(prompt=prompt, max_tokens=max_tokens, temperature=1.8, top_p=0.9)
```

</details>
</div>

In [ ]:
visualisiere_sampling(modell=chat_modell)

## Attention is all you need!

Ein Sprachmodell zerlegt einen Text in Tokens und wandelt jedes Token in einen Vektor um, also eine lange Liste von Zahlen, die seine Bedeutung repräsentiert. Das Problem: Ein einzelnes Token kennt seinen Kontext nicht. „Bank" kann ein Sitzmöbel oder ein Geldinstitut sein, und erst die umgebenden Wörter entscheiden das. **Attention** ist der Mechanismus, mit dem jedes Token Informationen von anderen Tokens im Text einsammelt. Eingeführt wurde er 2017 im Artikel *„Attention Is All You Need"*, der Grundlage aller heutigen Sprachmodelle.

**Die Idee: Suchanfrage, Schlagwort, Inhalt.** Aus jedem Token-Vektor berechnet das Modell (durch Multiplikation mit gelernten Matrizen $W_Q, W_K, W_V$) drei neue Vektoren:

- **Query** ($q$): Wonach sucht dieses Token?
- **Key** ($k$): Was hat dieses Token zu bieten?
- **Value** ($v$): Welche Information gibt es weiter, wenn es ausgewählt wird?

Wie gut eine Query zu einem Key passt, misst das Skalarprodukt $q \cdot k$. Für alle Tokens gleichzeitig geschrieben ergibt sich die zentrale Formel:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Die Division durch $\sqrt{d_k}$ (die Länge der Key-Vektoren) hält die Zahlen in einem handlichen Bereich. Die Softmax-Funktion macht aus den Passungswerten **Gewichte zwischen 0 und 1, die sich zu 1 addieren**, ähnlich wie Prozentanteile. Jedes Token erhält am Ende einen gewichteten Mittelwert der Values aller Tokens, auf die es achtet. Genau diese Gewichte werden wir gleich sichtbar machen.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Kausale Attention.</summary>

Ein Sprachmodell lernt, das jeweils nächste Token vorherzusagen. Dürfte es dabei in die Zukunft schauen, wäre das gemogelt. Deshalb darf jedes Token nur auf sich selbst und auf frühere Tokens achten. Technisch werden die Werte für spätere Tokens vor der Softmax auf $-\infty$ gesetzt und erhalten so das Gewicht 0.

</details>
</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Multi-Head Attention.</summary>

Ein einzelner Attention-Mechanismus könnte nur eine Art von Beziehung abbilden. Deshalb arbeiten in jeder Schicht mehrere **Heads** parallel, jeder mit eigenen Matrizen $W_Q, W_K, W_V$. So kann sich ein Head z.B. auf Grammatik spezialisieren, ein anderer auf Wiederholungen. Ihre Ergebnisse werden zusammengeführt und an die nächste Schicht weitergegeben.

</details>
</div>

In [ ]:
prompt = "The capital of France is Paris."
chat_modell.zeige_attention(prompt, schicht = [-1])

**So lesen Sie die Visualisierung:**

- Oben sehen Sie eine kleine Kachel pro Head. Fahren Sie mit der Maus darüber oder klicken Sie, um den Head groß anzuzeigen.
- **Zeilen** sind die Tokens, die „hinschauen" (Query), **Spalten** die Tokens, auf die geschaut wird (Key). Je kräftiger die Farbe, desto höher das Gewicht.
- Das graue Dreieck oben rechts ist die Zukunft, die durch Causal Attention verdeckt ist.
- Viele Heads legen den Großteil ihrer Aufmerksamkeit auf das **erste Token**. Das ist eine Art Parkplatz: Hat ein Head an einer Stelle nichts zu tun, muss er seine Aufmerksamkeit trotzdem irgendwo hinlegen (die Gewichte müssen sich zu 1 addieren). Das erste Token ist von überall aus sichtbar, und das Modell lernt, dass Hinschauen dort „nichts bewirkt". Interessant sind daher vor allem die Heads, die *woanders* hinschauen.

### Typische Muster: Previous Token Heads

Ein **Previous Token Head** macht etwas sehr Einfaches: Jedes Token schaut fast ausschließlich auf das Token direkt davor. In der Visualisierung erkennt man das an einer kräftigen Linie **direkt unterhalb der Diagonalen**.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Warum ist das nützlich?</summary>

Nach diesem Head „weiß" jedes Token, welches Token ihm vorausging. Das ist wichtiger, als es klingt:

- **Wörter bestehen oft aus mehreren Tokens.** Ein seltener Name wie „Thornbury" wird z.B. in „Thorn" und „bury" zerlegt. Erst wenn „bury" weiß, dass „Thorn" davorstand, kann das Modell das ganze Wort verarbeiten.
- **Bedeutung hängt vom Nachbarn ab.** „York" nach „New" meint etwas anderes als „York" allein.
- **Er ist ein Baustein für komplexere Muster.** Heads späterer Schichten können auf die Information zurückgreifen, die der Previous Token Head eingesammelt hat. Das wichtigste Beispiel dafür sehen wir im nächsten Abschnitt.

</details>
</div>

In [ ]:
prompt = "todo"
chat_modell.zeige_attention(prompt, schicht = [-1])

### Typische Muster: Induction Heads

Ein **Induction Head** setzt eine Kopierregel um: *„Ich bin bei Token A. Wo kam A schon einmal vor? Welches Token folgte damals darauf? Dann sage ich das jetzt wieder voraus."* Kurz: Aus dem Muster **[A][B] … [A]** folgt die Vorhersage **[B]**.

Dafür arbeiten zwei Heads in verschiedenen Schichten zusammen:

1. Ein **Previous Token Head** in einer frühen Schicht hinterlegt bei Token B die Information „vor mir stand A".
2. Der **Induction Head** in einer späteren Schicht sucht beim zweiten A nach einer Stelle, deren Vorgänger A war. Er findet B und kopiert dessen Information.

Im folgenden Beispiel wird ein unsinniger Satz wiederholt. Den zweiten Durchgang kann das Modell nicht aus Weltwissen vorhersagen, nur durch Kopieren. Achten Sie auf einen **diagonalen Streifen in der zweiten Hälfte**, der um die Länge des Satzes versetzt ist: Das zweite „purple" schaut auf das erste „elephant", das zweite „seven" auf das erste „rusty" und so weiter.

In [ ]:
prompt = "The purple elephant juggled seven rusty spoons on the roof. The purple elephant juggled seven rusty spoons on the roof."
chat_modell.zeige_attention(prompt, schicht = [-1])

**Warum sind Induction Heads so wichtig?**

Ein Sprachmodell kennt aus dem Training nur, was in den Trainingsdaten stand. Trotzdem kann es mit Namen, Begriffen und Mustern umgehen, die es nur im aktuellen Text gesehen hat. Man nennt das **In-Context Learning**: Lernen aus dem Kontext, ohne dass sich das Modell selbst verändert. Induction Heads sind ein zentraler Mechanismus dafür:

- **Neue Namen und Begriffe:** Kommt in einem Text „Mr. Thornbury" vor, kann das Modell den Namen später korrekt vervollständigen, obwohl es diesen Menschen nie gesehen hat.
- **Muster aus Beispielen übernehmen:** Gibt man dem Modell im Prompt Beispiele vor (z.B. „Hund → dog, Katze → cat, Maus → ..."), hilft das Kopieren der Struktur bei der Fortsetzung.
- **Programmcode:** Variablennamen, die weiter oben definiert wurden, können später wieder aufgegriffen werden.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>In-Kontext Lernen</summary>

Forschende bei Anthropic haben 2022 beobachtet, dass Induction Heads während des Trainings **recht plötzlich** entstehen. Genau zu diesem Zeitpunkt springt auch die Fähigkeit des Modells zum In-Context Learning sprunghaft nach oben. Das ist ein starkes Indiz dafür, dass beides zusammenhängt.

</details>
</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Wiederholungsschleifen</summary>

Die Kehrseite: Dieselbe Kopierneigung trägt vermutlich dazu bei, dass kleine Modelle manchmal in **Wiederholungsschleifen** geraten. Hat ein Modell einen Satz einmal geschrieben, macht der Induction-Mechanismus es wahrscheinlicher, ihn erneut zu schreiben.

</details>
</div>

Im nächsten Beispiel sehen Sie den realistischen Anwendungsfall: Der seltene Name „Thornbury" wird in mehrere Tokens zerlegt. Beim zweiten „Thorn" schaut ein Induction Head auf das Token, das beim ersten Mal folgte, und kann so den Namen vervollständigen.

In [ ]:
prompt = "Mr. Thornbury opened the door. Everyone looked at Mr. Thorn"
chat_modell.zeige_attention(prompt, schicht = [-1])

## [Optional] System Prompts

Der System-Prompt steht ganz am Anfang des Gesprächs und legt fest, **wie sich das Modell verhalten soll**: welche Rolle es einnimmt, in welchem Ton es antwortet, welche Regeln es beachtet. Technisch ist er nichts Besonderes, sondern einfach Text mit der Rolle `system`. Das Modell wurde aber darauf trainiert, diesem Text besonderes Gewicht zu geben.

Wir stellen dieselbe Frage zweimal: einmal mit dem Standard-System-Prompt und einmal mit einem eigenen.

In [ ]:
chat_modell.gedaechtnis.loesche()
chat_modell.prompt_auto(prompt="What did you do today?")
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

In [ ]:
chat_modell.gedaechtnis.loesche()
chat_modell.gedaechtnis.neuer_eintrag(rolle="system", inhalt="You are a cowboy.")
chat_modell.prompt_auto(prompt="What did you do today?")
print(chat_modell.zeige_prompt(chat_modell.gedaechtnis))

**Beobachtung:** Mit dem Standard-System-Prompt antwortet das Modell als KI-Assistent und erklärt vermutlich, dass es keinen Tag erlebt hat. Mit „You are a cowboy.“ schlüpft es in die Rolle und erfindet einen Tag auf der Ranch. Ein einziger Satz verändert das gesamte Verhalten.

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Wofür werden System-Prompts in der Praxis genutzt?</summary>

Jede Chat-Anwendung arbeitet mit System-Prompts, die Nutzerinnen und Nutzer meist nicht zu sehen bekommen. Darin stehen z.B.:

- Rolle und Ton („Du bist ein freundlicher Kundenservice-Assistent.“)
- Regeln („Stelle keine medizinischen Diagnosen.“)
- aktuelle Informationen, die das Modell sonst nicht kennen kann, etwa das heutige Datum
- Beschreibungen von Werkzeugen, die das Modell benutzen darf, etwa eine Websuche

Für den Unterricht interessant: Mit einem System-Prompt lässt sich z.B. ein Tutor bauen, der **keine fertigen Lösungen verrät, sondern Denkanstöße gibt**. Probieren Sie es aus – kleine Modelle wie SmolLM2 halten sich allerdings nur eingeschränkt an solche Anweisungen:

```python
chat_modell.gedaechtnis.loesche()
chat_modell.gedaechtnis.neuer_eintrag(
    rolle="system",
    inhalt="You are a physics tutor. Never give the final answer. "
    "Instead, give a short hint that helps the student find it themselves.",
)
chat_modell.prompt_auto(prompt="How long does a stone take to fall 20 meters?")
```

</details>
</div>

<div style="background-color:#f5f5f5; color:#333333; border:1px solid #dddddd; border-radius:4px; padding:10px 15px; margin:10px 0;">
<details>
<summary>Ist ein System-Prompt eine sichere Grenze?</summary>

Nein. Der System-Prompt ist nur Text, und das Modell folgt ihm, weil es darauf trainiert wurde – nicht, weil es technisch dazu gezwungen ist. Mit geschickt formulierten Nachrichten lassen sich Modelle manchmal dazu bringen, ihre Anweisungen zu ignorieren (*Jailbreak*). Ähnliches kann passieren, wenn Anweisungen in Dokumenten oder Webseiten versteckt sind, die das Modell verarbeitet (*Prompt Injection*). Große Modelle sind dagegen deutlich robuster als kleine, aber nicht unfehlbar.

</details>
</div>